In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqs, freqz
from ipywidgets import FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# MATCHED Z-TRANSFORM
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.mz-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.mz-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.mz-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.mz-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.mz-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
    margin-bottom:6px;
}

.mz-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.mz-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14px !important;
}

.jupyter-widgets input{
    font-size:13.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="mz-root">

<div class="mz-header">
Matched Z-Transform: Mapping of Poles and Zeros
</div>

<div class="mz-doc">

<b>Purpose.</b>
This notebook demonstrates the matched Z-transform for an analog transfer function
containing both poles and a finite zero. In contrast to impulse invariance, the
matched Z-transform maps not only the poles, but also the finite zeros of the
analog filter.

<br><br>

<b>Example analog filter.</b>
Consider

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
H<sub>a</sub>(s) =
(s+1) /
[(s+2)(s²+2s+5)].
</b>
</div>

This filter has one finite zero and three poles. Therefore,

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
N = 3,
&nbsp;&nbsp;
M = 1,
&nbsp;&nbsp;
L = N-M = 2.
</b>
</div>

The finite poles and zeros are mapped according to

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
z = e<sup>sT<sub>s</sub></sup>.
</b>
</div>

The two zeros at <b>s = ∞</b> are mapped to

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>z = -1</b>
</div>

and therefore contribute the corrective factor <b>(z+1)²</b>.

<br><br>

<b>Normalization.</b>
The matched Z-transform does not automatically preserve the desired gain.
The digital numerator is therefore scaled so that

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
H<sub>d</sub>(1)=H<sub>a</sub>(0).
</b>
</div>

</div>

</div>
"""))

# ============================================================
# ANALOG FILTER
# ============================================================

analog_zero = np.array([-1.0+0j])

analog_poles = np.array([
    -2.0+0j,
    -1.0+2.0j,
    -1.0-2.0j
])

N = len(analog_poles)
M = len(analog_zero)
L = N-M

analog_num = np.real_if_close(np.poly(analog_zero)).astype(float)
analog_den = np.real_if_close(np.poly(analog_poles)).astype(float)

# ============================================================
# CONTROL
# ============================================================

Ts_slider = FloatSlider(value=0.50,min=0.10,max=1.00,step=0.01,description='Sampling period T:',continuous_update=True,readout_format='.2f',style={'description_width':'125px'},layout=Layout(width='420px'))

transform_title = HTML('<div class="mz-title" style="margin:0;">Transform parameter</div>',layout=Layout(width='170px'))

controls = HBox([
    transform_title,
    Ts_slider
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0',align_items='center'))

info_output = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# ANALOG FREQUENCY RESPONSE
# ============================================================

Omega = np.logspace(-2,2,4000)

Omega_plot,Ha = freqs(analog_num,analog_den,worN=Omega)

analog_mag_dB = 20*np.log10(np.maximum(np.abs(Ha),1e-12))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.6))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. ANALOG POLES AND ZERO
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)
ax1.axvline(0,color='black',linewidth=0.8)
ax1.axvspan(-3.5,0,alpha=0.05)

ax1.plot(np.real(analog_poles),np.imag(analog_poles),'rx',markersize=8,markeredgewidth=1.8,label='Analog poles')
ax1.plot(np.real(analog_zero),np.imag(analog_zero),'bo',markersize=6,markerfacecolor='none',markeredgewidth=1.5,label='Analog zero')

ax1.set_xlim(-3.5,0.8)
ax1.set_ylim(-3.0,3.0)

ax1.set_title('Analog Poles and Zero in the s-Plane')
ax1.set_xlabel(r'$\Re\{s\}$')
ax1.set_ylabel(r'$\Im\{s\}$')

ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 2. DIGITAL POLES AND ZEROS
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

ax2.axhline(0,color='black',linewidth=0.8)
ax2.axvline(0,color='black',linewidth=0.8)

ax2.plot(np.cos(theta),np.sin(theta),'--',linewidth=1.1,label='Unit circle')

digital_poles_plot, = ax2.plot([],[],'rx',markersize=8,markeredgewidth=1.8,label='Mapped poles')

digital_zero_plot, = ax2.plot([],[],'bo',markersize=6,markerfacecolor='none',markeredgewidth=1.5,label='Mapped finite zero')

infinity_zeros_plot, = ax2.plot([],[],'gs',markersize=5.5,label=r'Zeros from $s=\infty$')

ax2.set_xlim(-1.2,1.2)
ax2.set_ylim(-1.2,1.2)
ax2.set_aspect('equal',adjustable='box')

ax2.set_title('Matched Z-Transform Poles and Zeros')
ax2.set_xlabel(r'$\Re\{z\}$')
ax2.set_ylabel(r'$\Im\{z\}$')

ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 3. ANALOG MAGNITUDE RESPONSE
# ============================================================

ax3.semilogx(Omega_plot,analog_mag_dB,color='red',linewidth=1.4,label='Analog magnitude response')

ax3.set_xlim(1e-2,1e2)
ax3.set_ylim(-100,10)

ax3.set_title('Analog Magnitude Response')
ax3.set_xlabel(r'Analog frequency $\Omega$')
ax3.set_ylabel('Magnitude (dB)')

ax3.grid(True,which='both',linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),frameon=False)

# ============================================================
# 4. DIGITAL MAGNITUDE RESPONSE
# ============================================================

digital_mag_line, = ax4.plot([],[],color='red',linewidth=1.4,label='Matched Z-transform magnitude response')

ax4.set_xlim(0,1)
ax4.set_ylim(-100,10)

ax4.set_title('Digital Magnitude Response')
ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax4.set_ylabel('Magnitude (dB)')

ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.30,hspace=0.62)

# ============================================================
# UPDATE
# ============================================================

def update_matched_z(change=None):

    T = Ts_slider.value

    # --------------------------------------------------------
    # Finite poles and zero
    # --------------------------------------------------------

    digital_poles = np.exp(analog_poles*T)

    digital_finite_zero = np.exp(analog_zero*T)

    # --------------------------------------------------------
    # Zeros at infinity
    # --------------------------------------------------------

    infinity_zeros = -np.ones(L,dtype=complex)

    all_digital_zeros = np.concatenate([
        digital_finite_zero,
        infinity_zeros
    ])

    # --------------------------------------------------------
    # Digital transfer function
    # --------------------------------------------------------

    a = np.real_if_close(np.poly(digital_poles)).astype(float)

    b_unscaled = np.real_if_close(np.poly(all_digital_zeros)).astype(float)

    # --------------------------------------------------------
    # Gain normalization
    # Hd(1) = Ha(0)
    # --------------------------------------------------------

    Ha0 = np.polyval(analog_num,0.0)/np.polyval(analog_den,0.0)

    Hd0_unscaled = np.polyval(b_unscaled,1.0)/np.polyval(a,1.0)

    K = np.real_if_close(Ha0/Hd0_unscaled).item()

    b = K*b_unscaled

    # --------------------------------------------------------
    # Digital frequency response
    # --------------------------------------------------------

    omega,Hd = freqz(b,a,worN=32768)

    Hd_dB = 20*np.log10(np.maximum(np.abs(Hd),1e-12))

    # --------------------------------------------------------
    # Update graphics
    # --------------------------------------------------------

    digital_poles_plot.set_data(
        np.real(digital_poles),
        np.imag(digital_poles)
    )

    digital_zero_plot.set_data(
        np.real(digital_finite_zero),
        np.imag(digital_finite_zero)
    )

    infinity_zeros_plot.set_data(
        np.real(infinity_zeros),
        np.imag(infinity_zeros)
    )

    digital_mag_line.set_data(
        omega/np.pi,
        Hd_dB
    )

    # --------------------------------------------------------
    # Numerical information
    # --------------------------------------------------------

    pole_text = '<br>'.join([
        f'<b>{np.real(z):.6f} {np.imag(z):+.6f}j</b>'
        for z in digital_poles
    ])

    zero_text = '<br>'.join([
        f'<b>{np.real(z):.6f} {np.imag(z):+.6f}j</b>'
        for z in digital_finite_zero
    ])

    info_output.value = f"""
    <div class="mz-root">

    <div class="mz-box">

    <div class="mz-title">Current matched Z-transform</div>

    <div class="mz-cols">

    <div class="mz-col">
    Sampling period:<br>
    <b>T = {T:.2f}</b>
    <br><br>
    Number of poles:<br>
    <b>N = {N}</b>
    </div>

    <div class="mz-col">
    Number of finite zeros:<br>
    <b>M = {M}</b>
    <br><br>
    Zeros at infinity:<br>
    <b>L = N-M = {L}</b>
    </div>

    <div class="mz-col">
    Mapped finite zero:<br>
    {zero_text}
    <br><br>
    Additional digital zeros:<br>
    <b>z = -1</b>, multiplicity <b>{L}</b>
    </div>

    <div class="mz-col">
    Mapped digital poles:<br>
    {pole_text}
    </div>

    </div>

    </div>

    <div class="mz-box">

    <div class="mz-title">Gain normalization</div>

    The digital numerator is scaled so that

    <div style="text-align:center;font-size:15.5px;margin:7px 0;">
    <b>
    H<sub>d</sub>(1)=H<sub>a</sub>(0).
    </b>
    </div>

    Current normalization constant:

    <div style="text-align:center;font-size:15.5px;margin:7px 0;">
    <b>K = {K:.8f}</b>
    </div>

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENT
# ============================================================

Ts_slider.observe(update_matched_z,names='value')

# ============================================================
# INITIAL NUMERICAL INFORMATION
# ============================================================

update_matched_z()

# ============================================================
# DISPLAY
# ============================================================

display(info_output)

# Slider deliberately immediately above the plots
display(controls)

display(fig.canvas)

# Initial redraw after attaching the canvas
update_matched_z()